# 5.4 GMM Distribution Analysis of Latent Dimensions

This notebook analyzes the distribution of each *active* latent dimension to determine if it is better modeled as:
- **Unimodal (Single Gaussian)**: Likely representing a continuous factor or noise.
- **Bimodal (Mixture of 2 Gaussians)**: Potentially representing a discrete subtype or categorical factor.

## Methodology
1. **Load Active Dimensions**: Use the same filtering logic as 5.1.1 (KL > Threshold).
2. **Fit GMMs**: For each dimension $d$, fit GMM($k=1$) and GMM($k=2$).
3. **Model Selection**: Compare models using the **Bayesian Information Criterion (BIC)**.
   - $\Delta BIC = BIC(k=1) - BIC(k=2)$
   - If $\Delta BIC > 10$ (positive evidence), the dimension is classified as **Bimodal**.
4. **Visualization**: Plot histograms with GMM overlays for the top bimodal candidates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4e' % x)

## 1. Data Loading & Active Unit Filtering

In [ ]:
# --- PARAMETERS ---
TARGET_BETA = "3.00e-05"
KL_THRESHOLD = 0.05
LATENT_CSV = "output/final_train_combined_vae_data.csv"
KL_CSV_PATH = f"output/Experiments/BetaScanVAE/beta_scan_results/beta_{TARGET_BETA}/kl_divergence.csv"

# Load Data
if os.path.exists(LATENT_CSV):
    df_latent = pd.read_csv(LATENT_CSV)
    print(f"Loaded Latent Data: {df_latent.shape}")
else:
    raise FileNotFoundError(f"Latent data not found at {LATENT_CSV}")

# Determine Active Dimensions
DIM_COLS = [c for c in df_latent.columns if c.startswith('latent_')]
active_features = []

if os.path.exists(KL_CSV_PATH):
    print(f"Loading KL Divergence from: {KL_CSV_PATH}")
    df_kl = pd.read_csv(KL_CSV_PATH)
    for _, row in df_kl.iterrows():
        if row['mean_kl'] > KL_THRESHOLD:
            active_features.append(row['dimension'])
else:
    print("Using Latent Variance as proxy for activity.")
    variances = df_latent[DIM_COLS].var()
    for dim, var in variances.items():
        if var > KL_THRESHOLD:
            active_features.append(dim)

ACTIVE_FEATURES = sorted(active_features, key=lambda x: int(x.split('_')[1]))
print(f"Active Dimensions: {len(ACTIVE_FEATURES)} / {len(DIM_COLS)}")

## 2. GMM Fitting & Bimodality Detection

In [ ]:
gmm_results = []

print("Fitting GMMs for Active Dimensions...")

for dim in ACTIVE_FEATURES:
    X = df_latent[[dim]].values # Reshape for sklearn
    
    # Fit k=1 (Unimodal)
    gmm1 = GaussianMixture(n_components=1, random_state=42)
    gmm1.fit(X)
    bic1 = gmm1.bic(X)
    
    # Fit k=2 (Bimodal)
    gmm2 = GaussianMixture(n_components=2, random_state=42)
    gmm2.fit(X)
    bic2 = gmm2.bic(X)
    
    # Calculate Delta BIC
    # Positive Delta -> BIC(1) > BIC(2) -> k=2 is better (lower BIC)
    delta_bic = bic1 - bic2
    
    is_bimodal = delta_bic > 10 # Threshold for strong evidence
    
    gmm_results.append({
        'Dimension': dim,
        'BIC_1': bic1,
        'BIC_2': bic2,
        'Delta_BIC': delta_bic,
        'Is_Bimodal': is_bimodal,
        'Means_k2': gmm2.means_.flatten(),
        'Weights_k2': gmm2.weights_.flatten()
    })

df_gmm = pd.DataFrame(gmm_results)
df_gmm = df_gmm.sort_values('Delta_BIC', ascending=False).reset_index(drop=True)

print("\nTop 10 Bimodal Candidates (Highest Delta BIC):")
print(df_gmm[['Dimension', 'Delta_BIC', 'Is_Bimodal']].head(10))

## 3. Visualization of Top Bimodal Dimensions

In [ ]:
top_bimodal = df_gmm[df_gmm['Is_Bimodal']].head(6)['Dimension'].tolist()

if len(top_bimodal) > 0:
    plt.figure(figsize=(15, 10))
    for i, dim in enumerate(top_bimodal):
        plt.subplot(2, 3, i+1)
        
        # Plot Histogram
        sns.histplot(df_latent[dim], kde=False, stat="density", bins=50, color='skyblue', label='Data')
        
        # Plot GMM Overlay
        X_plot = np.linspace(df_latent[dim].min(), df_latent[dim].max(), 1000).reshape(-1, 1)
        gmm2 = GaussianMixture(n_components=2, random_state=42).fit(df_latent[[dim]])
        logprob = gmm2.score_samples(X_plot)
        pdf = np.exp(logprob)
        
        plt.plot(X_plot, pdf, '-r', linewidth=2, label='GMM (k=2)')
        plt.title(f"{dim}\n$\Delta BIC = {df_gmm[df_gmm['Dimension']==dim]['Delta_BIC'].values[0]:.1f}$", fontsize=10)
        plt.xlabel("Latent Value")
        plt.legend()
        
    plt.tight_layout()
    plt.savefig("output/gmm_top_bimodal_dimensions.png")
    plt.show()
    print("Saved plot to output/gmm_top_bimodal_dimensions.png")
else:
    print("No dimensions classified as bimodal.")

## 4. Save Results

In [ ]:
output_path_csv = "output/gmm_dimension_analysis.csv"
df_gmm.to_csv(output_path_csv, index=False)
print(f"Full GMM Analysis results saved to: {output_path_csv}")

# Export list of Bimodal Dimensions for next notebook
bimodal_dims = df_gmm[df_gmm['Is_Bimodal']]['Dimension'].tolist()
print(f"\nIdentified {len(bimodal_dims)} Bimodal Dimensions.")